# Video Editing Demo

This tutorial demonstrates NeoVerse's capability to edit specific regions of a video using a binary mask and a text prompt, while keeping the rest of the scene intact.

**Workflow:** Given an input video and a corresponding mask video (white = region to edit, black = keep), NeoVerse reconstructs the 3D scene, blanks out the masked region, and synthesizes new content guided by the text prompt.

Two cases are shown:
- **Case 1:** Teapot scene — edit the masked object guided by a new text description
- **Case 2:** Driving scene — replace the masked vehicle with a different car

## Configuration

- `use_lora`: Enable 4-step distilled LoRA for fast inference
- `num_frames`: 81 frames for smooth motion (NeoVerse default)
- `alpha_threshold`: Controls which pixels get repainted by diffusion (1.0 = repaint all masked regions)
- `static_scene`: Set to False for videos with dynamic scenes; True for static scenes
- `resize_mode`: "center_crop" preserves aspect ratio while fitting target resolution

In [ ]:
import torch
import os
import numpy as np
from torchvision.transforms import functional as F

import sys
sys.path.append("..")
from diffsynth.pipelines.wan_video_neoverse import WanVideoNeoVersePipeline
from diffsynth import save_video
from diffsynth.utils.auxiliary import CameraTrajectory, load_video, homo_matrix_inverse


use_lora = True
model_path = "../models"
reconstructor_path = "../models/NeoVerse/reconstructor.ckpt"
low_vram = False
seed = 42
num_frames = 81
width = 560
height = 336
resize_mode = "center_crop"
static_scene = False
alpha_threshold = 1.0

num_inference_steps = 4 if use_lora else 50
cfg_scale = 1.0 if use_lora else 5.0

In [ ]:
lora_path = os.path.join(
    model_path,
    "NeoVerse/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors"
) if use_lora else None

torch.manual_seed(seed)
np.random.seed(seed)

# Load model
print(f"Loading model from {model_path}...")
pipe = WanVideoNeoVersePipeline.from_pretrained(
    local_model_path=model_path,
    reconstructor_path=reconstructor_path,
    lora_path=lora_path,
    lora_alpha=1.0,
    device="cuda",
    torch_dtype=torch.bfloat16,
    enable_vram_management=low_vram,
)
print("Model loaded!")

# Case 1: Teapot Scene

Edit the masked region in the teapot video using a text prompt. The mask marks the area to be synthesized; the surrounding scene is preserved.

In [ ]:
input_path = "../examples/videos/video_editing_case1.mp4"
input_mask_path = "../examples/videos/video_editing_case1_mask.mp4"
output_path = "../outputs/video_editing_case1.mp4"
prompt = "A transparent glass teapot is pouring out loose tea leaves and pale green tea."
input_video = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
input_masks = load_video(input_mask_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
input_masks = [mask.convert("L") for mask in input_masks]
cam_traj = CameraTrajectory.from_predefined("static")
static_flag = static_scene

with torch.no_grad():
    device = pipe.device
    height, width = input_video[0].size[1], input_video[0].size[0]
    views = {
        "img": torch.stack([F.to_tensor(image)[None] for image in input_video], dim=1).to(device),
        "mask": torch.stack([F.to_tensor(mask)[None] for mask in input_masks], dim=1).to(device) > 0,
        "is_target": torch.zeros((1, len(input_video)), dtype=torch.bool, device=device),
    }
    if static_flag:
        views["is_static"] = torch.ones((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.zeros((1, len(input_video)), dtype=torch.int64, device=device)
    else:
        views["is_static"] = torch.zeros((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.arange(0, len(input_video), dtype=torch.int64, device=device).unsqueeze(0)

    # Low-VRAM: load reconstructor to GPU before use
    if pipe.vram_management_enabled:
        pipe.reconstructor.to(device)

    with torch.amp.autocast("cuda", dtype=pipe.torch_dtype):
        predictions = pipe.reconstructor(views, is_inference=True, use_motion=False)

    # Low-VRAM: offload reconstructor back to CPU
    if pipe.vram_management_enabled:
        pipe.reconstructor.cpu()
        torch.cuda.empty_cache()

    gaussians = predictions["splats"]
    K = predictions["rendered_intrinsics"][0]
    input_cam2world = predictions["rendered_extrinsics"][0]
    timestamps = predictions["rendered_timestamps"][0]

    if static_flag:
        K = K[:1].repeat(len(cam_traj), 1, 1)
        timestamps = timestamps[:1].repeat(len(cam_traj))

    # Apply per-trajectory zoom_ratio
    ratio = torch.linspace(1, cam_traj.zoom_ratio, K.shape[0], device=device)
    K_zoomed = K.clone()
    K_zoomed[:, 0, 0] *= ratio
    K_zoomed[:, 1, 1] *= ratio

    target_cam2world = cam_traj.c2w.to(device)
    if cam_traj.mode == "relative" and not static_flag:
        target_cam2world = input_cam2world @ target_cam2world
    target_world2cam = homo_matrix_inverse(target_cam2world)
    target_rgb, target_depth, target_alpha = pipe.reconstructor.gs_renderer.rasterizer.forward(
        gaussians,
        render_viewmats=[target_world2cam],
        render_Ks=[K_zoomed],
        render_timestamps=[timestamps],
        sh_degree=0, width=width, height=height,
    )
    # editing mode
    target_rgb = views["img"].permute(0, 1, 3, 4, 2)
    target_mask = 1 - views["mask"].permute(0, 1, 3, 4, 2).float()
    target_rgb = target_rgb * target_mask

    wrapped_data = {
        "source_views": views,
        "target_rgb": target_rgb,
        "target_depth": target_depth,
        "target_mask": target_mask,
        "target_poses": target_cam2world.unsqueeze(0),
        "target_intrs": K_zoomed.unsqueeze(0),
    }
    generated_frames = pipe(
        prompt=prompt,
        negative_prompt="",
        seed=seed, rand_device=pipe.device,
        height=height, width=width, num_frames=len(target_cam2world),
        cfg_scale=cfg_scale, num_inference_steps=num_inference_steps, tiled=False,
        **wrapped_data,
    )
    save_video(generated_frames, output_path, fps=16)
print(f"Done! Output saved to: {output_path}")

# Case 2: Driving Scene

Replace the masked vehicle in the driving video with a red sports sedan using a different text prompt.

In [ ]:
input_path = "../examples/videos/video_editing_case2.mp4"
input_mask_path = "../examples/videos/video_editing_case2_mask.mp4"
output_path = "../outputs/video_editing_case2.mp4"
prompt = "A red sports sedan with a low profile is driving on the road."
input_video = load_video(input_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
input_masks = load_video(input_mask_path, num_frames,
                    resolution=(width, height),
                    resize_mode=resize_mode,
                    static_scene=static_scene)
input_masks = [mask.convert("L") for mask in input_masks]
cam_traj = CameraTrajectory.from_predefined("static")
static_flag = static_scene

with torch.no_grad():
    device = pipe.device
    height, width = input_video[0].size[1], input_video[0].size[0]
    views = {
        "img": torch.stack([F.to_tensor(image)[None] for image in input_video], dim=1).to(device),
        "mask": torch.stack([F.to_tensor(mask)[None] for mask in input_masks], dim=1).to(device) > 0,
        "is_target": torch.zeros((1, len(input_video)), dtype=torch.bool, device=device),
    }
    if static_flag:
        views["is_static"] = torch.ones((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.zeros((1, len(input_video)), dtype=torch.int64, device=device)
    else:
        views["is_static"] = torch.zeros((1, len(input_video)), dtype=torch.bool, device=device)
        views["timestamp"] = torch.arange(0, len(input_video), dtype=torch.int64, device=device).unsqueeze(0)

    # Low-VRAM: load reconstructor to GPU before use
    if pipe.vram_management_enabled:
        pipe.reconstructor.to(device)

    with torch.amp.autocast("cuda", dtype=pipe.torch_dtype):
        predictions = pipe.reconstructor(views, is_inference=True, use_motion=False)

    # Low-VRAM: offload reconstructor back to CPU
    if pipe.vram_management_enabled:
        pipe.reconstructor.cpu()
        torch.cuda.empty_cache()

    gaussians = predictions["splats"]
    K = predictions["rendered_intrinsics"][0]
    input_cam2world = predictions["rendered_extrinsics"][0]
    timestamps = predictions["rendered_timestamps"][0]

    if static_flag:
        K = K[:1].repeat(len(cam_traj), 1, 1)
        timestamps = timestamps[:1].repeat(len(cam_traj))

    # Apply per-trajectory zoom_ratio
    ratio = torch.linspace(1, cam_traj.zoom_ratio, K.shape[0], device=device)
    K_zoomed = K.clone()
    K_zoomed[:, 0, 0] *= ratio
    K_zoomed[:, 1, 1] *= ratio

    target_cam2world = cam_traj.c2w.to(device)
    if cam_traj.mode == "relative" and not static_flag:
        target_cam2world = input_cam2world @ target_cam2world
    target_world2cam = homo_matrix_inverse(target_cam2world)
    target_rgb, target_depth, target_alpha = pipe.reconstructor.gs_renderer.rasterizer.forward(
        gaussians,
        render_viewmats=[target_world2cam],
        render_Ks=[K_zoomed],
        render_timestamps=[timestamps],
        sh_degree=0, width=width, height=height,
    )
    # editing mode
    target_rgb = views["img"].permute(0, 1, 3, 4, 2)
    target_mask = 1 - views["mask"].permute(0, 1, 3, 4, 2).float()
    target_rgb = target_rgb * target_mask

    wrapped_data = {
        "source_views": views,
        "target_rgb": target_rgb,
        "target_depth": target_depth,
        "target_mask": target_mask,
        "target_poses": target_cam2world.unsqueeze(0),
        "target_intrs": K_zoomed.unsqueeze(0),
    }
    generated_frames = pipe(
        prompt=prompt,
        negative_prompt="",
        seed=seed, rand_device=pipe.device,
        height=height, width=width, num_frames=len(target_cam2world),
        cfg_scale=cfg_scale, num_inference_steps=num_inference_steps, tiled=False,
        **wrapped_data,
    )
    save_video(generated_frames, output_path, fps=16)
print(f"Done! Output saved to: {output_path}")

## Results

Two edited videos are generated in `../outputs/`:

- `video_editing_case1.mp4`: Teapot scene with the masked region resynthesized
- `video_editing_case2.mp4`: Driving scene with the masked vehicle replaced

The unmasked regions remain visually consistent with the original video, demonstrating NeoVerse's ability to perform localized, prompt-guided video editing.